# 🎭 Nexa — Diffusion-Based Face Swapper**IP-Adapter FaceID + Stable Diffusion 1.5** face-swap pipeline.### Requirements- Google Colab with **GPU runtime** (T4 recommended)- Upload your `source.jpg` (face to use) and `target.jpg` (image to swap into)> **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
#@title 1️⃣ Install Dependenciesimport subprocess, sys, os# Check GPUgpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],                          capture_output=True, text=True)print(f"GPU: {gpu_info.stdout.strip()}")# Install core packages first!pip install -q torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118 2>&1 | tail -2!pip install -q diffusers transformers accelerate huggingface_hub safetensors scipy 2>&1 | tail -2!pip install -q insightface 2>&1 | tail -2!pip install -q typer rich imageio[ffmpeg] ffmpeg-python opencv-python-headless 2>&1 | tail -2# Handle onnxruntime conflict!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null!pip install -q onnxruntime-gpu 2>&1 | tail -2# Verifyimport torchprint(f"\n✅ PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"   GPU: {torch.cuda.get_device_name(0)}")    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
#@title 2️⃣ Install FFmpeg (for video processing)!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1!ffmpeg -version | head -1print("✅ FFmpeg ready")

In [ ]:
#@title 3️⃣ Setup Nexa Packageimport os# Check if nexa folder existsNEXA_DIR = "/content/nexa"if os.path.exists(NEXA_DIR):    print(f"✅ Nexa found at {NEXA_DIR}")else:    print("⚠️  Nexa folder not found at /content/nexa")    print("    Please upload the nexa folder to /content/")    print("    Or upload nexa-project.zip and run:")    print("    !unzip -o /content/nexa-project.zip -d /content/")    raise FileNotFoundError("Upload nexa folder first")os.chdir(NEXA_DIR)# Install nexa as editable package!pip install -e . 2>&1 | tail -3# Verifyimport nexaprint(f"\n✅ Nexa v{nexa.__version__} installed")# Quick import testfrom nexa.models.swapper import FaceSwapper, MLPProjModel, LoRAAttnProcessor, LoRAIPAttnProcessorprint("✅ All core modules imported successfully")

In [ ]:
#@title 4️⃣ Upload Source & Target Imagesfrom google.colab import filesfrom IPython.display import display, Image as IPImageprint("Upload your SOURCE face image (the face identity to use):")uploaded_source = files.upload()source_name = list(uploaded_source.keys())[0]source_path = f"/content/{source_name}"with open(source_path, 'wb') as f:    f.write(uploaded_source[source_name])print(f"✅ Source saved: {source_path}")display(IPImage(source_path, width=256))print("\nUpload your TARGET image (the image where faces will be swapped):")uploaded_target = files.upload()target_name = list(uploaded_target.keys())[0]target_path = f"/content/{target_name}"with open(target_path, 'wb') as f:    f.write(uploaded_target[target_name])print(f"✅ Target saved: {target_path}")display(IPImage(target_path, width=256))

In [ ]:
#@title 5️⃣ Run Face Swap (CLI Mode)#@markdown ### Configurationsteps = 20  #@param {type:"slider", min:10, max:40, step:5}strength = 0.65  #@param {type:"slider", min:0.3, max:0.9, step:0.05}guidance_scale = 5.0  #@param {type:"slider", min:2.0, max:10.0, step:0.5}ip_scale = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}enhancer = "none"  #@param ["none", "gfpgan"]import osos.chdir("/content/nexa")output_path = "/content/output.jpg"cmd = f"python -m nexa.main"cmd += f" --source {source_path}"cmd += f" --target {target_path}"cmd += f" --output {output_path}"cmd += f" --steps {steps}"cmd += f" --strength {strength}"cmd += f" --guidance-scale {guidance_scale}"cmd += f" --ip-scale {ip_scale}"cmd += f" --gpu"if enhancer != "none":    cmd += f" --enhancer {enhancer}"print(f"Running:\n{cmd}\n")!{cmd}# Display resultfrom IPython.display import display, Image as IPImageif os.path.exists(output_path):    print("\n✅ Face swap complete!")    display(IPImage(output_path, width=512))else:    print("❌ Output not found — check errors above.")

In [ ]:
#@title 6️⃣ Alternative: Run via Python API (more control)import osos.chdir("/content/nexa")import torchfrom IPython.display import display, Image as IPImage# Import Nexa pipelinefrom nexa.core.pipeline import NexaPipeline# Initialize pipeline (this loads all models)pipeline = NexaPipeline(    model_id="runwayml/stable-diffusion-v1-5",    device="cuda",    steps=20,    enhancer_name=None,  # Set to "gfpgan" for enhancement    threshold=0.6,    ip_scale=1.0,    strength=0.65,    guidance_scale=5.0,)# Run swapresult_path = pipeline.process_image_single(    source_path=source_path,    target_path=target_path,    output_path="/content/output_api.jpg",)print(f"✅ Result saved to: {result_path}")display(IPImage(str(result_path), width=512))

In [ ]:
#@title 7️⃣ Video Face Swap (Optional)#@markdown Upload a video file and run face swap on all frames.import osos.chdir("/content/nexa")from google.colab import filesprint("Upload your TARGET video:")uploaded_video = files.upload()video_name = list(uploaded_video.keys())[0]video_path = f"/content/{video_name}"with open(video_path, 'wb') as f:    f.write(uploaded_video[video_name])print(f"✅ Video saved: {video_path}")video_output = "/content/output_video.mp4"cmd = f"python -m nexa.main"cmd += f" --source {source_path}"cmd += f" --target {video_path}"cmd += f" --output {video_output}"cmd += f" --steps 15 --gpu"print(f"Running:\n{cmd}\n")!{cmd}if os.path.exists(video_output):    print(f"\n✅ Video saved: {video_output}")    files.download(video_output)else:    print("❌ Video output not found.")

In [ ]:
#@title 8️⃣ Side-by-Side Comparisonimport cv2import numpy as npfrom IPython.display import display, Image as IPImageimport osoutput_file = "/content/output.jpg"if not os.path.exists(output_file):    output_file = "/content/output_api.jpg"if os.path.exists(target_path) and os.path.exists(output_file):    target_img = cv2.imread(target_path)    output_img = cv2.imread(output_file)    # Resize to same height    h = min(target_img.shape[0], output_img.shape[0], 512)    target_img = cv2.resize(target_img, (int(target_img.shape[1] * h / target_img.shape[0]), h))    output_img = cv2.resize(output_img, (int(output_img.shape[1] * h / output_img.shape[0]), h))    # Add labels    cv2.putText(target_img, 'ORIGINAL', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)    cv2.putText(output_img, 'SWAPPED', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)    # Concatenate side by side    comparison = np.hstack([target_img, output_img])    comp_path = "/content/comparison.jpg"    cv2.imwrite(comp_path, comparison)    display(IPImage(comp_path, width=1024))else:    print("Run a face swap first!")

In [ ]:
#@title 9️⃣ Download Resultsfrom google.colab import filesimport osfor f in ["/content/output.jpg", "/content/output_api.jpg", "/content/comparison.jpg", "/content/output_video.mp4"]:    if os.path.exists(f):        print(f"Downloading: {f}")        files.download(f)print("\n✅ Done!")

In [ ]:
#@title 🔧 Debug: Check Model Loading (run if face swap fails)import torchimport osos.chdir("/content/nexa")print("=== Environment ===")print(f"PyTorch: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB / {torch.cuda.mem_get_info()[1] / 1024**3:.1f} GB")print("\n=== Testing InsightFace ===")try:    from nexa.models.analyzer import FaceAnalyzer    analyzer = FaceAnalyzer()    print("✅ InsightFace loaded")except Exception as e:    print(f"❌ InsightFace failed: {e}")print("\n=== Testing IP-Adapter FaceID Checkpoint ===")try:    from nexa.models.manager import download_ip_adapter_faceid    ckpt_path = download_ip_adapter_faceid()    sd = torch.load(str(ckpt_path), map_location="cpu")    print(f"✅ Checkpoint loaded: {list(sd.keys())}")    print(f"   image_proj keys: {list(sd['image_proj'].keys())}")    ip_keys = list(sd['ip_adapter'].keys())    print(f"   ip_adapter keys: {len(ip_keys)} total")    indices = sorted(set(int(k.split('.')[0]) for k in ip_keys))    print(f"   Processor indices: {len(indices)} processors")except Exception as e:    print(f"❌ Checkpoint failed: {e}")print("\n=== Testing SD1.5 Pipeline ===")try:    from diffusers import StableDiffusionImg2ImgPipeline    print("✅ diffusers imported")except Exception as e:    print(f"❌ diffusers failed: {e}")print("\n=== Testing FaceSwapper Init ===")try:    from nexa.models.swapper import FaceSwapper    swapper = FaceSwapper(device="cuda", steps=5)    print("✅ FaceSwapper initialized!")    print(f"   Processors: {len(swapper.pipe.unet.attn_processors)}")except Exception as e:    print(f"❌ FaceSwapper failed: {e}")    import traceback    traceback.print_exc()